# **Imports**

In [2]:
import scipy.io as sio
import numpy as np
import statistical_features
import matplotlib.pyplot as plt
import scipy.io as scio

from joblib import Parallel, delayed

# **Configuration**

In [3]:
MAT_PATH = r"subject1.mat"
WINDOW_SIZE = 20
wavelet = statistical_features.WaveletPreprocessor(level=7, wavelet='db4')
N_JOBS = -1

# **Load Dataset**

In [4]:
mat = sio.loadmat(MAT_PATH)
save_ECG = mat["save_ECG"]
print(f"Dataset Shape : {save_ECG.shape}")

Dataset Shape : (588579, 762)


# **Split Columns**

In [5]:
ecg_cycles = save_ECG[:, :760]
glucose = save_ECG[:, 760]
pkl_ids = save_ECG[:, 761]

print(f"ECG Cycles : {ecg_cycles.shape}")
print(f"Glucose    : {glucose.shape}")
print(f"PKL IDs    : {pkl_ids.shape}")

ECG Cycles : (588579, 760)
Glucose    : (588579,)
PKL IDs    : (588579,)


In [6]:
a = ecg_cycles[1:20]
b = a.reshape(-1)
b.shape, len(ecg_cycles)

((14440,), 588579)

# **Build Feature Headers**

In [7]:
feature_headers = []

for feature_name in statistical_features.FEATURE_ORDER:
    for signal_name in statistical_features.SIGNAL_ORDER:
        feature_headers.append(f"{feature_name}_{signal_name}")

feature_headers.append("Label")
feature_headers.append("PKL_ID")

# **Window Processing Function**

In [8]:
def process_window(i):

    start = i * WINDOW_SIZE
    end = start + WINDOW_SIZE

    ecg_window = ecg_cycles[start:end]

    segment = ecg_window.reshape(-1)

    feature_database = wavelet.extract_wavelet_features(segment)

    feature_vector = statistical_features.build_feature_vector(feature_database)

    return (feature_vector, glucose[start], pkl_ids[start])

# **Feature Extraction**

In [9]:
num_windows = len(ecg_cycles) // WINDOW_SIZE

results = Parallel(n_jobs=-1, prefer="processes")(
    delayed(process_window)(i)
    for i in range(num_windows)
)

# **Organize Results**

In [10]:
feature_Matrix = [r[0] for r in results]
labels = [r[1] for r in results]
window_pkl = [r[2] for r in results]

# Convert to NumPy
feature_Matrix = np.asarray(feature_Matrix, dtype=np.float32)
labels = np.asarray(labels, dtype=np.float32)
window_pkl = np.asarray(window_pkl, dtype=np.float32)

# **Save to MATLAB file**

In [15]:
labels = labels.reshape(-1,1)
window_pkl = window_pkl.reshape(-1,1)
feature_dataset = np.hstack((feature_Matrix, labels, window_pkl))
scio.savemat("Subject1_Temporal_Feature.mat",{"feature_dataset": feature_dataset, "feature_headers": feature_headers})